# Qiskit Fall Fest 2026, Challenge 3: Catch the eavesdropper

Opens Wed, Nov 18, 2026 at 6:00 PM. Closes Sun, Nov 22, 2026 at 11:59 PM. Both times are Clemson time (US Eastern).

**The task.** Spot which BB84 key exchanges were intercepted.

Start from Wednesday's notebook. Its Lab 3 took the records of twelve key exchanges, computed the error rate of each one, and labeled it clean or intercepted with one threshold. This notebook gives you 40 new records. On these the threshold alone is not enough, and finding what else to look at is the challenge.

How to use this notebook:

- Run the three code cells of the Setup section first. This challenge needs no quantum computer and no account, so leave the three switches as they are.
- Read "The task in detail". It says what the records are, what you are told about them, and how your answer is scored.
- Run the cells under "The tools". Their code is complete. They load the records and apply Wednesday's rule to them.
- Do your own work under "Your attempt".
- Turn in the line that the notebook prints. The steps are under "Turn it in". Only three lines per person count, so read that part before you send anything.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## The task in detail

### The records

BB84 is the key exchange of Wednesday's session. Alice sends qubits to Bob. For each qubit she picks a bit, 0 or 1, and a basis. A basis is one of two ways to prepare or to measure a qubit, called Z and X. Bob measures each qubit in a basis that he picks at random too. A qubit that is measured in the basis it was prepared in gives the bit that was sent. Measured in the other basis, it gives 0 or 1 with equal chance.

Afterwards Alice and Bob keep the positions where their bases match. Those bits are the sifted key. On it, Bob's result should equal Alice's bit. The error rate is the fraction of kept positions where the two differ.

You get the records of 40 exchanges, named `ex01` to `ex40`. Each exchange has 1,000 qubits and four arrays of 0s and 1s, as in Wednesday's Lab 3:

- `alice`: Alice's bits.
- `a_bases`: Alice's bases, 0 for the Z basis and 1 for the X basis.
- `b_bases`: Bob's bases.
- `bob`: Bob's results.

The arrays are in the order in which the qubits were sent. Entry 0 belongs to the first qubit and entry 999 to the last.

Be clear about where the records come from. They are not a recording from a real device. A program generated them from the rules of BB84, as `make_batch` did in Wednesday's notebook. Every participant gets the same 40 records.

### What you are told about the records

- **Noise.** Device noise flips each of Bob's results on its own. Every exchange has one fixed probability for such a flip, somewhere between 1% and 10%. Within an exchange that probability is the same at every position and in both bases.
- **Eve.** An eavesdropper, called Eve as on Wednesday, intercepted some of the exchanges. When she is there she uses intercept and resend, the attack on Wednesday's slides: she measures a qubit in a basis of her choice and sends on what she measured. She does not always work in the same way. She may listen to every qubit, to a share of the qubits spread over the whole exchange, or only during one stretch of it. She may pick her basis at random for each qubit or always measure in the same one.
- **How many.** Exactly 20 of the 40 exchanges were intercepted.
- **The promise.** Every intercepted exchange leaves a pattern in its errors that noise of this kind cannot produce. An exchange without such a pattern is clean, however high its error rate.

### The rules

- The challenge is scored individually. Your labels have to be your own work. Do not post or send your line or your labels to anyone before the close, in Discord or anywhere else.
- No quantum computer is needed. The challenge is an analysis of recorded data.
- You turn in one label for each exchange: 1 for intercepted and 0 for clean.
- Only your first three well-formed lines count, and the best of them is your score. A fourth line is ignored. A line is well-formed when it has the form of the line this notebook prints: `QFF26-C3 labels=` followed by 40 characters, each of them 0 or 1.

### Your score

Labeling the 40 exchanges by tossing a coin gets 20 of them right on average. So the score counts what you get right beyond 20:

$$\text{score} = \max\left(0, \; 5 \times (\text{correct} - 20)\right)$$

Here "correct" is the number of exchanges with the right label. 30 right labels score 50, and 40 score 100. How the event uses the scores is explained on the event page.

### What to expect

Wednesday's rule, unchanged, gets 28 of the 40 labels right, which is a score of 40. You do not need to spend one of your three lines to find that out. A full score is possible: every interception can be found from the records and the description above.

## The tools

The code in this part is complete. Run each cell and read what it does.

The first cell holds the 40 records and loads them. The records are packed into one long line of text, `PACKED`. You do not need to read it. `load_batch` checks that the text is complete and unchanged, then unpacks it into `batch`. That is a dictionary like the one in Wednesday's Lab 3: `batch["ex01"]["alice"]` is the array of Alice's bits in the first exchange.

How the check works: a hash is a short fingerprint computed from a text. If one character of the text changes, the fingerprint changes too. The cell compares the fingerprint of `PACKED` with the one it must have.

In [ ]:
import base64
import hashlib

import numpy as np

EXCHANGES = 40                                     # exchanges in the batch
QUBITS = 1000                                      # qubits in each exchange
FIELDS = ("alice", "bob", "a_bases", "b_bases")    # the four arrays of one exchange
PACKED = "..."
PACKED_LENGTH = 26668
PACKED_SHA256 = "25431058be3a68f1df30f91c904d645f005280c1302af376f1f1309dabcda59b"


def load_batch(text):
    """Unpack the recorded exchanges into a dictionary: the name of an exchange, then its four arrays."""
    intact = len(text) == PACKED_LENGTH and hashlib.sha256(text.encode()).hexdigest() == PACKED_SHA256
    assert intact, "The text in PACKED was changed. Download the challenge notebook again."
    bits = np.unpackbits(np.frombuffer(base64.b64decode(text), dtype=np.uint8))
    bits = bits[:EXCHANGES * len(FIELDS) * QUBITS].reshape(EXCHANGES, len(FIELDS), QUBITS)
    return {f"ex{k + 1:02d}": dict(zip(FIELDS, bits[k].astype(int))) for k in range(EXCHANGES)}


batch = load_batch(PACKED)
print("exchanges in the batch:", len(batch), "| the first is", list(batch)[0], "and the last is", list(batch)[-1])
print("arrays in one exchange:", list(batch["ex01"]), "with", len(batch["ex01"]["alice"]), "entries each")

The second cell is the code of Wednesday's Lab 3 with a few additions: it stores its labels in the dictionary `wednesday`, prints every error rate with three decimals and the number of kept bits, and ends with the number of exchanges that it labeled intercepted.

- `qber(ex)` keeps the positions where the bases match, compares Alice's bits with Bob's results there, and returns the fraction that differ. That is the error rate. QBER is short for quantum bit error rate.
- `threshold = 0.11` is the value from Wednesday's seminar: the error rate above which a key is thrown away.
- The loop prints, for each exchange, its name, its error rate, Wednesday's label and the number of kept bits.

In [ ]:
def qber(ex):
    keep = ex["a_bases"] == ex["b_bases"]
    a, b = ex["alice"][keep], ex["bob"][keep]
    return np.mean(a != b)


threshold = 0.11
wednesday = {}                                     # Wednesday's label for each exchange: 1 intercepted, 0 clean
for name, ex in batch.items():
    rate = qber(ex)
    if rate > threshold:
        label = "intercepted"
    else:
        label = "clean"
    wednesday[name] = int(rate > threshold)
    print(name, format(rate, ".3f"), label, "| kept bits:", int(np.sum(ex["a_bases"] == ex["b_bases"])))
print("labeled intercepted:", sum(wednesday.values()), "of", len(wednesday))

Wednesday's rule labels 8 of the 40 exchanges intercepted. Their error rates are all above 0.15. Noise of at most 10% does not produce that, so these eight really were intercepted.

The description says that 20 exchanges were intercepted. Twelve interceptions are therefore hidden among the 32 exchanges below the threshold, where the highest error rate is 0.094. No threshold on the error rate alone sorts those 32 out. By the description, a clean exchange with strong noise and an intercepted exchange can have the same error rate.

The third cell puts the 40 error rates on the number line of Wednesday's Lab 3.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 2.2))
ax.plot([0, 28], [0, 0], color="0.4", lw=1)                                        # the number line
for name, ex in batch.items():
    rate = qber(ex)
    if rate > threshold:
        ax.plot(100 * rate, 0, "s", color="tab:red", markersize=8, alpha=0.5)      # a square: labeled intercepted
    else:
        ax.plot(100 * rate, 0, "o", color="tab:blue", markersize=8, alpha=0.5)     # a circle: labeled clean
ax.plot([100 * threshold, 100 * threshold], [-0.25, 0.25], color="black", ls="--")  # the threshold
ax.plot([], [], "o", color="tab:blue", label="labeled clean by Wednesday's rule")   # two entries for the legend
ax.plot([], [], "s", color="tab:red", label="labeled intercepted")
ax.legend(loc="upper center", ncol=2, frameon=False)
ax.set_xlim(-2, 31)
ax.set_ylim(-0.4, 1.0)
ax.set_yticks([])
ax.set_xticks([0, 5, 10, 15, 20, 25])
ax.set_xlabel("error rate on the sifted key (percent)")
ax.set_title("The 40 exchanges of the challenge batch")
for side in ("left", "right", "top"):
    ax.spines[side].set_visible(False)
plt.show()

The eight squares lie to the right of the dashed threshold. Markers that lie on top of each other look darker. On Wednesday a wide empty stretch around the threshold made the decision safe. Here the 32 circles reach up to 9.4 percent, and twelve of them are interceptions.

The fourth cell checks the format of a set of labels and writes the line you turn in.

- `problems(labels)` lists the reasons why a set of labels cannot be turned in. An empty list means that the format is right. It says nothing about whether the labels are correct. Nothing in this notebook can tell you that.
- `submission_line(labels)` writes the line: the 40 labels in the order `ex01` to `ex40`. It stops with an error if `problems` finds something.

In [ ]:
NAMES = [f"ex{k + 1:02d}" for k in range(EXCHANGES)]


def is_label(value):
    """True for the number 0 or 1, also written as 0.0 or 1.0 or as False or True."""
    return isinstance(value, (bool, int, float, np.integer, np.floating, np.bool_)) and value in (0, 1)


def problems(labels):
    """Return the list of reasons why these labels cannot be turned in. An empty list means the format is right."""
    if not isinstance(labels, dict):
        return ["labels must be a dictionary: the name of an exchange, then 1 or 0"]
    found = []
    if set(labels) != set(NAMES):
        found.append("labels needs exactly the 40 names ex01 to ex40")
    if not all(is_label(value) for value in labels.values()):
        found.append("every label must be the number 1 (intercepted) or 0 (clean)")
    return found


def submission_line(labels):
    """The one line of text that you turn in. Stops with the reasons if problems(labels) finds any."""
    found = problems(labels)
    if found:
        raise ValueError("These labels cannot be turned in: " + "; ".join(found))
    return "QFF26-C3 labels=" + "".join(str(int(labels[name])) for name in NAMES)


lab_rule = {name: int(qber(batch[name]) > 0.11) for name in NAMES}      # Wednesday's rule with Wednesday's threshold, whatever was changed above
print("format problems of Wednesday's labels:", problems(lab_rule))
print("Wednesday's labels as a line:", submission_line(lab_rule))

That line is well-formed, and it scores 40, as "What to expect" said. Sending it would use up one of your three lines.

## Your attempt

From here on the work is yours. The next cell holds your rule. As given it is Wednesday's rule. Change `my_rule`, or build `labels` in any other way you like, as long as `labels` ends up as a dictionary that gives every exchange a 1 or a 0.

Wednesday's rule looks at one number for each exchange. The description of the records points to three questions that it never asks:

- **Where in an exchange do the errors fall?** The arrays are in the order in which the qubits were sent.
- **In which basis do the errors fall?**
- **How large does a difference have to be before it is more than chance?** An exchange has about 500 kept bits. An error rate measured on $N$ bits scatters around its true value $p$ by about $\sqrt{p(1-p)/N}$. For $p = 0.08$ and $N = 250$ that is 0.017. Wednesday's Lab 2 called this scatter shot noise.

Two numbers help you test your own reasoning: there are 40 exchanges, and exactly 20 of them were intercepted. Nobody tells you which of your labels were wrong.

In [ ]:
def my_rule(ex):
    """Return 1 if you judge this exchange intercepted and 0 if you judge it clean. As given: Wednesday's rule."""
    return int(qber(ex) > 0.11)


labels = {name: my_rule(ex) for name, ex in batch.items()}
print("labeled intercepted:", sum(labels.values()), "of", len(labels))

The next cell checks the format of `labels` and prints your line. It cannot tell you whether a label is right. It does compare the number of exchanges that you marked with the 20 that the description promises. Run the cell above again after every change to `my_rule`: the next cell prints the line for the labels that the cell above made last.

In [ ]:
trouble = problems(labels)
if trouble:
    print("These labels cannot be turned in: " + "; ".join(trouble))
else:
    marked = sum(int(labels[name]) for name in NAMES)
    if marked != 20:
        print("Your labels mark", marked, "exchanges as intercepted. The batch has exactly 20, so at least",
              abs(marked - 20), "of your labels are wrong.")
    if all(int(labels[name]) == lab_rule[name] for name in NAMES):
        print("These are the labels of Wednesday's rule. They score 40.")
    print("Your line:")
    print(submission_line(labels))
    print("Only your first three well-formed lines count. Send this one only when you are ready.")

## Turn it in

1. Copy the whole line that starts with `QFF26-C3`. Do not change anything in it.
2. Open the submission form: [https://forms.gle/zRgtCHGC682mVUDm7](https://forms.gle/zRgtCHGC682mVUDm7)
3. Enter the email you registered with and a leaderboard name, paste the line, and send. The name is shown on the leaderboards of the event page. Type the same name every time: the boards keep the name you typed first in any of the five forms, and only an organizer can change it.

Only your first three well-formed lines count, and the best of them is your score. A fourth line is ignored, and sending the same line again uses up one of the three. Nobody tells you which labels were wrong: the board on the event page shows only your points. So check your reasoning before you send a line, not after.

## If something breaks

| What you see | What to do |
|---|---|
| The setup cell stops and asks for a restart | Restart the kernel, then run the notebook again from the top. It installed the tested packages |
| "The text in PACKED was changed" | The long line of text was edited or cut off. Download the challenge notebook again |
| "labels needs exactly the 40 names ex01 to ex40" | Build `labels` from `batch.items()`, as the cell does when you first open the notebook |
| `RuntimeWarning: Mean of empty slice` and the value `nan` | You computed an error rate on a selection that holds no kept position. Check the selection |
| Your labels mark more or fewer than 20 exchanges | The batch has exactly 20 interceptions. Look again at the exchanges close to your own thresholds |
| `NameError: name 'qber' is not defined`, or the same for `batch`, `problems`, `lab_rule` or `labels` | Run the cells under "The tools" first, from the top, then the first code cell under "Your attempt" |
| "every label must be the number 1 (intercepted) or 0 (clean)" | A label is something other than 0 or 1, for example a text, an error rate or an array. Make `my_rule` return `int(...)` of a yes-or-no test |

## Where to go from here

- Every challenge stays open until Sun, Nov 22 at 11:59 PM, and a full score counts whenever it arrives before then. The first ten people to reach 100 also get a bonus (see the event page), but only three lines count, so send a line only when your reasoning has convinced you.
- Stuck on the notebook itself? Ask in Discord between sessions (discord.gg/FBNzRcxMr), also if your score on the board looks wrong. Ask about the code, not about the labels: the challenge is scored individually.